In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # EX · ML Anomaly Detection para Trust & Safety
# MAGIC
# MAGIC ## YelpConnect
# MAGIC
# MAGIC ### Objetivo
# MAGIC
# MAGIC Detectar comportamientos estadísticamente atípicos en Reviews
# MAGIC utilizando Machine Learning no supervisado.
# MAGIC
# MAGIC El modelo utiliza features previamente construidas en:
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_features`
# MAGIC
# MAGIC y genera:
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_ml_score`
# MAGIC
# MAGIC ## Importante
# MAGIC
# MAGIC Una anomalía estadística:
# MAGIC
# MAGIC - NO representa fraude confirmado.
# MAGIC - NO bloquea automáticamente una Review.
# MAGIC - NO modifica el risk_score actual.
# MAGIC - constituye una señal experimental complementaria.
# MAGIC
# MAGIC El motor determinístico R001-R006 continúa siendo
# MAGIC el mecanismo T&S operacional.

# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    ArrayType,
    DoubleType
)

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.clustering import (
    KMeans
)

from pyspark.ml.evaluation import (
    ClusteringEvaluator
)

from pyspark.ml.functions import (
    vector_to_array
)

import uuid
import json

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOLD_SCHEMA = "yelp_gold"
GOV_SCHEMA = "yelp_gov"


FEATURE_TABLE = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}."
    "slv_review_features"
)


ML_SCORE_TABLE = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}."
    "slv_review_ml_score"
)


ML_RUN_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "ml_model_run"
)


RISK_TABLE = (
    f"{CATALOG}."
    f"{GOLD_SCHEMA}."
    "gld_review_risk"
)


AI_TABLE = (
    f"{CATALOG}."
    f"{GOLD_SCHEMA}."
    "gld_review_ai_enrichment"
)


ML_RUN_ID = str(
    uuid.uuid4()
)

ML_RUN_TS = (
    datetime
    .now(timezone.utc)
)


# ============================================================
# PARÁMETROS DEL MODELO
# ============================================================

K_CLUSTERS = 8

SEED = 42

MAX_ITER = 30

TRAIN_PERCENT = 80

# Top ~0.5% más alejados dentro de cada cluster
ANOMALY_PERCENTILE = 0.995


print("=" * 72)
print("YELPCONNECT - ML ANOMALY DETECTION")
print("=" * 72)

print(
    f"Run ID              : {ML_RUN_ID}"
)

print(
    f"Features            : {FEATURE_TABLE}"
)

print(
    f"Output              : {ML_SCORE_TABLE}"
)

print(
    f"K clusters          : {K_CLUSTERS}"
)

print(
    f"Train               : {TRAIN_PERCENT}%"
)

print(
    f"Anomaly percentile  : {ANOMALY_PERCENTILE}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # 3. Features utilizadas
# MAGIC
# MAGIC El modelo usa señales estructuradas previamente construidas.
# MAGIC
# MAGIC No utiliza directamente el texto de la Review.
# MAGIC
# MAGIC Esto complementa el notebook `01_analisis_ia_resenas`,
# MAGIC que sí analiza texto no estructurado.

# COMMAND ----------

FEATURE_COLUMNS = [

    # Diferencia entre rating de review
    # y rating agregado del negocio
    "rating_deviation",

    # Antigüedad de la cuenta
    "account_age_days",

    # Actividad temporal del usuario
    "user_reviews_30m",
    "user_reviews_1h",
    "user_reviews_24h",

    # Actividad general
    "user_review_count",

    # Concentración usuario-negocio
    "user_business_concentration",

    # Repetición exacta/normalizada del texto
    "text_repeat_count",

    # Actividad anómala del negocio
    "business_spike_ratio"
]


print(
    "Features ML:"
)

for feature in FEATURE_COLUMNS:

    print(
        f" - {feature}"
    )


# COMMAND ----------

# ============================================================
# 4. VALIDAR FUENTE
# ============================================================

if not spark.catalog.tableExists(
    FEATURE_TABLE
):

    raise Exception(
        f"""
        ❌ No existe:

        {FEATURE_TABLE}

        Ejecutar previamente:

        Feature Engineering Trust & Safety.
        """
    )


feature_schema = {
    field.name
    for field
    in spark.table(
        FEATURE_TABLE
    ).schema.fields
}


required_columns = {

    "review_id",
    "business_id",
    "user_id",

    *FEATURE_COLUMNS
}


missing_columns = (
    required_columns
    - feature_schema
)


if missing_columns:

    raise Exception(
        f"""
        ❌ Faltan columnas requeridas
        para ML:

        {sorted(missing_columns)}
        """
    )


print(
    "✔ Feature table disponible."
)

print(
    "✔ Features requeridas disponibles."
)


# COMMAND ----------

# ============================================================
# 5. PREPARAR DATASET ML
# ============================================================

source = (
    spark
    .table(
        FEATURE_TABLE
    )
)


review_count = (
    source
    .count()
)


distinct_review_count = (
    source

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Reviews           : "
    f"{review_count:,}"
)

print(
    f"Review ID únicos  : "
    f"{distinct_review_count:,}"
)


if review_count != distinct_review_count:

    raise Exception(
        """
        ❌ El grano de slv_review_features
        no es 1 Review.
        """
    )


# COMMAND ----------

# ============================================================
# 6. CAST EXPLÍCITO A DOUBLE
# ============================================================

dataset = (
    source

    .select(

        "review_id",
        "business_id",
        "user_id",

        *[
            F.col(
                feature
            )
            .cast(
                "double"
            )
            .alias(
                feature
            )

            for feature
            in FEATURE_COLUMNS
        ]
    )
)


display(
    dataset.limit(10)
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Train set determinístico
# MAGIC
# MAGIC Se utiliza aproximadamente 80% para entrenar el modelo.
# MAGIC
# MAGIC La separación se hace mediante hash de `review_id`,
# MAGIC evitando que una nueva ejecución produzca otro conjunto
# MAGIC aleatorio completamente diferente.

# COMMAND ----------

dataset = (
    dataset

    .withColumn(
        "_train_bucket",

        F.pmod(
            F.xxhash64(
                "review_id"
            ),

            F.lit(
                100
            )
        )
    )
)


training = (
    dataset

    .filter(
        F.col(
            "_train_bucket"
        )
        <
        TRAIN_PERCENT
    )
)


training_count = (
    training
    .count()
)


print(
    f"Dataset completo : "
    f"{review_count:,}"
)

print(
    f"Training         : "
    f"{training_count:,}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Pipeline de Machine Learning
# MAGIC
# MAGIC Etapas:
# MAGIC
# MAGIC 1. Imputación de valores nulos.
# MAGIC 2. Vectorización.
# MAGIC 3. Estandarización.
# MAGIC 4. Clustering mediante K-Means.
# MAGIC
# MAGIC K-Means no clasifica fraude.
# MAGIC
# MAGIC Se utiliza para aprender grupos de comportamiento
# MAGIC y posteriormente medir qué tan lejos queda una Review
# MAGIC respecto del centro de su cluster.

# COMMAND ----------

# ============================================================
# 8.1 IMPUTER
# ============================================================

IMPUTED_COLUMNS = [
    f"{feature}__imputed"

    for feature
    in FEATURE_COLUMNS
]


imputer = (
    Imputer(

        inputCols=FEATURE_COLUMNS,

        outputCols=IMPUTED_COLUMNS
    )

    .setStrategy(
        "median"
    )
)


# ============================================================
# 8.2 VECTOR ASSEMBLER
# ============================================================

assembler = (
    VectorAssembler(

        inputCols=IMPUTED_COLUMNS,

        outputCol="_features_raw"
    )
)


# ============================================================
# 8.3 STANDARD SCALER
# ============================================================

scaler = (
    StandardScaler(

        inputCol="_features_raw",

        outputCol="_features_scaled",

        withMean=True,

        withStd=True
    )
)


# ============================================================
# 8.4 K-MEANS
# ============================================================

kmeans = (
    KMeans(

        featuresCol="_features_scaled",

        predictionCol="ml_cluster_id",

        k=K_CLUSTERS,

        seed=SEED,

        maxIter=MAX_ITER
    )
)


# ============================================================
# 8.5 PIPELINE
# ============================================================

pipeline = (
    Pipeline(

        stages=[

            imputer,

            assembler,

            scaler,

            kmeans
        ]
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Entrenar modelo

# COMMAND ----------

model = (
    pipeline
    .fit(
        training
    )
)


print(
    "✔ Modelo K-Means entrenado."
)


# COMMAND ----------

# ============================================================
# 10. SCOREAR LAS 400K REVIEWS
# ============================================================

scored_base = (
    model

    .transform(
        dataset
    )
)


scored_count = (
    scored_base
    .count()
)


if scored_count != review_count:

    raise Exception(
        f"""
        ❌ Pérdida de registros durante scoring.

        Source : {review_count}
        Scored : {scored_count}
        """
    )


print(
    f"✔ Reviews scoreadas: "
    f"{scored_count:,}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Calidad del clustering
# MAGIC
# MAGIC `silhouette` permite medir separación/cohesión de clusters.
# MAGIC
# MAGIC No representa precisión de detección de fraude.
# MAGIC
# MAGIC Es únicamente una métrica de calidad del clustering.

# COMMAND ----------

evaluator = (
    ClusteringEvaluator(

        featuresCol="_features_scaled",

        predictionCol="ml_cluster_id",

        metricName="silhouette",

        distanceMeasure="squaredEuclidean"
    )
)


silhouette = (
    evaluator
    .evaluate(
        scored_base
    )
)


print(
    f"Silhouette score: "
    f"{silhouette:.4f}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Obtener centroides del modelo

# COMMAND ----------

kmeans_model = (
    model
    .stages[-1]
)


cluster_centers = (
    kmeans_model
    .clusterCenters()
)


center_schema = StructType([

    StructField(
        "ml_cluster_id",
        IntegerType(),
        False
    ),

    StructField(
        "ml_cluster_center",
        ArrayType(
            DoubleType()
        ),
        False
    )
])


center_rows = [

    (
        int(cluster_id),

        [
            float(value)
            for value
            in center
        ]
    )

    for cluster_id, center
    in enumerate(
        cluster_centers
    )
]


centers_df = (
    spark.createDataFrame(

        center_rows,

        schema=center_schema
    )
)


display(
    centers_df
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 13. Distancia de cada Review a su centroide
# MAGIC
# MAGIC Cuanto mayor sea la distancia:
# MAGIC
# MAGIC mayor es la diferencia respecto al patrón aprendido
# MAGIC del cluster al que pertenece la Review.

# COMMAND ----------

scored = (

    scored_base

    .withColumn(
        "_scaled_array",

        vector_to_array(
            "_features_scaled"
        )
    )

    .join(
        F.broadcast(
            centers_df
        ),

        on="ml_cluster_id",

        how="left"
    )

    .withColumn(
        "ml_distance",

        F.sqrt(

            F.expr(
                """
                aggregate(
                    zip_with(
                        _scaled_array,
                        ml_cluster_center,
                        (x, y) -> pow(x - y, 2)
                    ),
                    CAST(0 AS DOUBLE),
                    (acc, x) -> acc + x
                )
                """
            )
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Umbral de anomalía por cluster
# MAGIC
# MAGIC Utilizamos el percentil 99.5 de distancia de cada cluster.
# MAGIC
# MAGIC Una Review es anómala si su distancia supera el umbral
# MAGIC correspondiente a su propio cluster.
# MAGIC
# MAGIC Esto evita utilizar un único umbral para grupos
# MAGIC con dispersión diferente.

# COMMAND ----------

cluster_thresholds = (

    scored

    .groupBy(
        "ml_cluster_id"
    )

    .agg(

        F.count(
            "*"
        )
        .alias(
            "cluster_reviews"
        ),

        F.expr(
            f"""
            percentile_approx(
                ml_distance,
                {ANOMALY_PERCENTILE},
                10000
            )
            """
        )
        .alias(
            "ml_cluster_threshold"
        )
    )
)


display(
    cluster_thresholds
    .orderBy(
        "ml_cluster_id"
    )
)


# COMMAND ----------

# ============================================================
# 15. CALCULAR SCORE DE ANOMALÍA
# ============================================================

scored = (

    scored

    .join(
        cluster_thresholds,

        on="ml_cluster_id",

        how="left"
    )

    .withColumn(

        "ml_anomaly_score",

        F.when(

            F.col(
                "ml_cluster_threshold"
            )
            >
            0,

            F.col(
                "ml_distance"
            )
            /
            F.col(
                "ml_cluster_threshold"
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )

    .withColumn(

        "ml_is_anomaly",

        F.col(
            "ml_distance"
        )
        >
        F.col(
            "ml_cluster_threshold"
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Interpretación de `ml_anomaly_score`
# MAGIC
# MAGIC Aproximadamente:
# MAGIC
# MAGIC - `< 1.0` → dentro del rango esperado del cluster.
# MAGIC - `> 1.0` → supera el umbral de anomalía.
# MAGIC
# MAGIC No es una probabilidad.
# MAGIC
# MAGIC Ejemplo:
# MAGIC
# MAGIC `ml_anomaly_score = 1.50`
# MAGIC
# MAGIC significa que la distancia observada es aproximadamente
# MAGIC 1.5 veces el umbral definido para ese cluster.

# COMMAND ----------

# ============================================================
# 16. PRODUCTO FINAL ML
# ============================================================

ml_output = (

    scored

    .select(

        "review_id",
        "business_id",
        "user_id",

        "ml_cluster_id",

        F.round(
            F.col(
                "ml_distance"
            ),
            6
        )
        .alias(
            "ml_distance"
        ),

        F.round(
            F.col(
                "ml_cluster_threshold"
            ),
            6
        )
        .alias(
            "ml_cluster_threshold"
        ),

        F.round(
            F.col(
                "ml_anomaly_score"
            ),
            6
        )
        .alias(
            "ml_anomaly_score"
        ),

        "ml_is_anomaly",

        F.lit(
            ML_RUN_ID
        )
        .alias(
            "ml_run_id"
        ),

        F.lit(
            "KMEANS_DISTANCE"
        )
        .alias(
            "ml_algorithm"
        ),

        F.lit(
            K_CLUSTERS
        )
        .alias(
            "ml_k"
        ),

        F.lit(
            ANOMALY_PERCENTILE
        )
        .alias(
            "ml_anomaly_percentile"
        ),

        F.current_timestamp()
        .alias(
            "ml_scored_at"
        )
    )
)


# COMMAND ----------

# ============================================================
# 17. PERSISTIR SCORE ML
# ============================================================

(
    ml_output

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        ML_SCORE_TABLE
    )
)


spark.sql(
    f"""
    COMMENT ON TABLE {ML_SCORE_TABLE}
    IS
    'Score experimental de anomalías ML para Reviews YelpConnect. '
    'Modelo no supervisado basado en distancia a centroides K-Means. '
    'Una anomalía representa comportamiento estadísticamente atípico '
    'y NO fraude confirmado.'
    """
)


print(
    f"✔ Score ML persistido: "
    f"{ML_SCORE_TABLE}"
)


# COMMAND ----------

# ============================================================
# 18. VALIDAR PRODUCTO
# ============================================================

ml_count = (
    spark.table(
        ML_SCORE_TABLE
    )
    .count()
)


ml_unique_reviews = (

    spark.table(
        ML_SCORE_TABLE
    )

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


if ml_count != review_count:

    raise Exception(
        f"""
        ❌ ML output incompleto.

        Esperado : {review_count}
        Actual   : {ml_count}
        """
    )


if ml_unique_reviews != review_count:

    raise Exception(
        """
        ❌ El producto ML no mantiene
        grano 1 Review.
        """
    )


print(
    f"✔ ML rows             : "
    f"{ml_count:,}"
)

print(
    f"✔ ML Review ID únicos : "
    f"{ml_unique_reviews:,}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 19. Distribución de clusters

# COMMAND ----------

display(

    spark.sql(
        f"""
        SELECT

            ml_cluster_id,

            COUNT(*) AS reviews,

            SUM(
                CASE

                    WHEN ml_is_anomaly

                    THEN 1

                    ELSE 0

                END
            ) AS anomalías,

            ROUND(
                AVG(
                    ml_anomaly_score
                ),
                4
            ) AS score_promedio,

            ROUND(
                MAX(
                    ml_anomaly_score
                ),
                4
            ) AS score_maximo

        FROM
            {ML_SCORE_TABLE}

        GROUP BY
            ml_cluster_id

        ORDER BY
            ml_cluster_id
        """
    )
)


# COMMAND ----------

# ============================================================
# 20. RESUMEN DE ANOMALÍAS
# ============================================================

ml_summary = (

    spark.sql(
        f"""
        SELECT

            COUNT(*) AS reviews,

            SUM(
                CASE

                    WHEN ml_is_anomaly

                    THEN 1

                    ELSE 0

                END
            ) AS anomalías,

            ROUND(
                100.0
                *
                SUM(
                    CASE

                        WHEN ml_is_anomaly

                        THEN 1

                        ELSE 0

                    END
                )

                /

                COUNT(*),

                4
            ) AS pct_anomalías

        FROM
            {ML_SCORE_TABLE}
        """
    )

    .first()
)


print()
print("=" * 72)
print("RESUMEN ML")
print("=" * 72)

print(
    f"Reviews    : "
    f"{ml_summary.reviews:,}"
)

print(
    f"Anomalías  : "
    f"{ml_summary.anomalías:,}"
)

print(
    f"% Anomalía : "
    f"{ml_summary.pct_anomalías}%"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # 21. Comparar ML vs reglas Trust & Safety
# MAGIC
# MAGIC Esto permite responder:
# MAGIC
# MAGIC > ¿Las anomalías detectadas estadísticamente ya estaban
# MAGIC > siendo identificadas por R001-R006?

# COMMAND ----------

if spark.catalog.tableExists(
    RISK_TABLE
):

    ml_vs_ts = (

        spark
        .table(
            ML_SCORE_TABLE
        )
        .alias(
            "ml"
        )

        .join(

            spark
            .table(
                RISK_TABLE
            )
            .select(

                "review_id",
                "signal_count",
                "risk_score",
                "signal_level",
                "investigation_priority",
                "requires_investigation"
            )
            .alias(
                "ts"
            ),

            on="review_id",

            how="inner"
        )
    )


    display(

        ml_vs_ts

        .groupBy(

            "ml_is_anomaly",

            (
                F.col(
                    "signal_count"
                )
                >
                0
            )
            .alias(
                "has_ts_signal"
            )
        )

        .agg(

            F.count(
                "*"
            )
            .alias(
                "reviews"
            )
        )

        .orderBy(

            F.desc(
                "ml_is_anomaly"
            ),

            F.desc(
                "has_ts_signal"
            )
        )
    )


else:

    print(
        "ℹ Gold Risk no disponible. "
        "Se omite comparación ML vs T&S."
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 22. Casos ML nuevos
# MAGIC
# MAGIC Reviews que:
# MAGIC
# MAGIC - ML considera anómalas.
# MAGIC - R001-R006 no activaron ninguna señal.
# MAGIC
# MAGIC Estas Reviews son interesantes para investigación exploratoria,
# MAGIC pero no deben considerarse fraude.

# COMMAND ----------

if spark.catalog.tableExists(
    RISK_TABLE
):

    display(

        ml_vs_ts

        .filter(

            (
                F.col(
                    "ml_is_anomaly"
                )
                ==
                True
            )

            &

            (
                F.col(
                    "signal_count"
                )
                ==
                0
            )
        )

        .select(

            "review_id",
            "business_id",
            "user_id",

            "ml_cluster_id",
            "ml_anomaly_score",

            "signal_count",
            "risk_score",
            "signal_level",
            "investigation_priority"
        )

        .orderBy(
            F.desc(
                "ml_anomaly_score"
            )
        )

        .limit(
            100
        )
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 23. Top anomalías

# COMMAND ----------

display(

    spark
    .table(
        ML_SCORE_TABLE
    )

    .orderBy(
        F.desc(
            "ml_anomaly_score"
        )
    )

    .limit(
        50
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 24. Comparación opcional ML + IA semántica + reglas
# MAGIC
# MAGIC Si se ejecutó previamente:
# MAGIC
# MAGIC `01_analisis_ia_resenas`
# MAGIC
# MAGIC podemos identificar casos donde coinciden:
# MAGIC
# MAGIC - anomalía ML;
# MAGIC - señal T&S;
# MAGIC - inconsistencia rating ↔ sentimiento.

# COMMAND ----------

if (
    spark.catalog.tableExists(
        AI_TABLE
    )
    and
    spark.catalog.tableExists(
        RISK_TABLE
    )
):

    triple_analysis = (

        spark
        .table(
            ML_SCORE_TABLE
        )
        .alias(
            "ml"
        )

        .join(

            spark
            .table(
                RISK_TABLE
            )
            .select(

                "review_id",
                "signal_count",
                "risk_score",
                "investigation_priority"
            )
            .alias(
                "ts"
            ),

            on="review_id",

            how="inner"
        )

        .join(

            spark
            .table(
                AI_TABLE
            )
            .select(

                "review_id",
                "ai_sentiment",
                "ai_topic",
                "rating_sentiment_inconsistent"
            )
            .alias(
                "ai"
            ),

            on="review_id",

            how="inner"
        )
    )


    display(

        triple_analysis

        .filter(
            F.col(
                "ml_is_anomaly"
            )
            ==
            True
        )

        .select(

            "review_id",

            "ml_anomaly_score",

            "signal_count",

            "risk_score",

            "investigation_priority",

            "ai_sentiment",

            "ai_topic",

            "rating_sentiment_inconsistent"
        )

        .orderBy(
            F.desc(
                "ml_anomaly_score"
            )
        )
    )


else:

    print(
        "ℹ IA semántica y/o Gold Risk "
        "no disponibles."
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 25. Registrar ejecución del modelo
# MAGIC
# MAGIC Se registra:
# MAGIC
# MAGIC - algoritmo;
# MAGIC - parámetros;
# MAGIC - features;
# MAGIC - centroides;
# MAGIC - métrica silhouette;
# MAGIC - volumen entrenado;
# MAGIC - umbral de anomalía.
# MAGIC
# MAGIC Esto permite trazabilidad básica del modelo.

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {ML_RUN_TABLE}
    (
        ml_run_id              STRING,
        ml_run_ts              TIMESTAMP,

        algorithm              STRING,

        k_clusters             INT,
        seed                   INT,
        max_iter               INT,

        train_percent          INT,
        anomaly_percentile     DOUBLE,

        source_table           STRING,
        source_rows            BIGINT,
        training_rows          BIGINT,

        features_json          STRING,
        cluster_centers_json   STRING,

        silhouette             DOUBLE,

        output_table           STRING,

        status                 STRING
    )
    USING DELTA
    """
)


features_json = (
    json.dumps(
        FEATURE_COLUMNS
    )
)


centers_json = (
    json.dumps(
        [
            [
                float(value)
                for value
                in center
            ]

            for center
            in cluster_centers
        ]
    )
)


run_df = (

    spark.createDataFrame(

        [
            (
                ML_RUN_ID,
                ML_RUN_TS,

                "KMEANS_DISTANCE",

                K_CLUSTERS,
                SEED,
                MAX_ITER,

                TRAIN_PERCENT,
                ANOMALY_PERCENTILE,

                FEATURE_TABLE,
                review_count,
                training_count,

                features_json,
                centers_json,

                float(
                    silhouette
                ),

                ML_SCORE_TABLE,

                "SUCCESS"
            )
        ],

        """
        ml_run_id STRING,
        ml_run_ts TIMESTAMP,

        algorithm STRING,

        k_clusters INT,
        seed INT,
        max_iter INT,

        train_percent INT,
        anomaly_percentile DOUBLE,

        source_table STRING,
        source_rows BIGINT,
        training_rows BIGINT,

        features_json STRING,
        cluster_centers_json STRING,

        silhouette DOUBLE,

        output_table STRING,

        status STRING
        """
    )
)


(
    run_df

    .write

    .format(
        "delta"
    )

    .mode(
        "append"
    )

    .saveAsTable(
        ML_RUN_TABLE
    )
)


print(
    f"✔ ML run registrado: "
    f"{ML_RUN_ID}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 26. Gobierno e interpretación
# MAGIC
# MAGIC ## Qué significa una anomalía ML
# MAGIC
# MAGIC Una Review marcada como `ml_is_anomaly = true` presenta
# MAGIC características alejadas del patrón aprendido para su cluster.
# MAGIC
# MAGIC Puede deberse a:
# MAGIC
# MAGIC - actividad temporal inusual;
# MAGIC - concentración atípica usuario-negocio;
# MAGIC - desviación fuerte de rating;
# MAGIC - repetición de texto;
# MAGIC - actividad anómala del negocio;
# MAGIC - combinación poco frecuente de varias características.
# MAGIC
# MAGIC ## Qué NO significa
# MAGIC
# MAGIC No significa que exista fraude confirmado.
# MAGIC
# MAGIC ## Uso previsto
# MAGIC
# MAGIC El ML sirve como:
# MAGIC
# MAGIC - señal exploratoria;
# MAGIC - apoyo para priorización;
# MAGIC - descubrimiento de patrones;
# MAGIC - complemento de R001-R006;
# MAGIC - fuente de hipótesis para nuevas reglas T&S.
# MAGIC
# MAGIC Antes de introducir esta señal al risk score productivo
# MAGIC deben existir validación humana, análisis de falsos positivos,
# MAGIC versionado y aprobación de gobierno.

# COMMAND ----------

# ============================================================
# 27. RESULTADO FINAL
# ============================================================

print()
print("=" * 72)
print("ML ANOMALY DETECTION FINALIZADO")
print("=" * 72)

print(
    f"Run ID          : "
    f"{ML_RUN_ID}"
)

print(
    f"Reviews         : "
    f"{ml_count:,}"
)

print(
    f"Anomalías       : "
    f"{ml_summary.anomalías:,}"
)

print(
    f"Anomaly %       : "
    f"{ml_summary.pct_anomalías}%"
)

print(
    f"Silhouette      : "
    f"{silhouette:.4f}"
)

print(
    f"Output          : "
    f"{ML_SCORE_TABLE}"
)

print()
print(
    "Una anomalía ML representa "
    "comportamiento estadísticamente atípico."
)

print(
    "NO representa fraude confirmado."
)

print("=" * 72)